In [1]:
from roboflow import Roboflow
rf = Roboflow(api_key="zDiJbZBMUgWUCSOo6Fvx")
workspace = rf.workspace("blazed")

print("Your projects:")
for project_name in workspace.projects():
    print(f"  - {project_name}")

loading Roboflow workspace...
Your projects:
  - blazed/chef1-m9q1c
  - blazed/chef2
  - blazed/cook-ppe-merged
  - blazed/final-work-nyuq2-fuyr5
  - blazed/final-work-nyuq2-qokdc
  - blazed/gloves-4nvvi-wcdnk
  - blazed/gp-fqlbs-jfcck
  - blazed/gp-fqlbs-rcqzy
  - blazed/hairnet-detection-vktcj-0jkon
  - blazed/hairnet-detection-vktcj-hok6l


In [2]:
projects = [
    "gp-fqlbs-jfcck",
    "gloves-4nvvi-wcdnk",
    "final-work-nyuq2-qokdc",
    "hairnet-detection-vktcj-0jkon",
    "chef2",
]

download_paths = []
for project_name in projects:
    project = rf.workspace("blazed").project(project_name)
    ds = project.version(1).download("yolov11", location=f"C:/Users/ziada/PycharmProjects/CVproject/{project_name}")
    download_paths.append(f"C:/Users/ziada/PycharmProjects/CVproject/{project_name}")
    print(f" Downloaded {project_name}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to C:/Users/ziada/PycharmProjects/CVproject/gp-fqlbs-jfcck in yolov11:: 100%|██████████| 6093/6093 [00:04<00:00, 1486.90it/s]

 Downloaded gp-fqlbs-jfcck
loading Roboflow workspace...


loading Roboflow project...



Extracting Dataset Version Zip to C:/Users/ziada/PycharmProjects/CVproject/gloves-4nvvi-wcdnk in yolov11:: 100%|██████████| 1007/1007 [00:00<00:00, 1678.62it/s]

...


Extracting Dataset Version Zip to C:/Users/ziada/PycharmProjects/CVproject/hairnet-detection-vktcj-0jkon in yolov11:: 100%|██████████| 7149/7149 [00:04<00:00, 1681.41it/s]

 Downloaded hairnet-detection-vktcj-0jkon
loading Roboflow workspace...


loading Roboflow project...



Extracting Dataset Version Zip to C:/Users/ziada/PycharmProjects/CVproject/chef2 in yolov11:: 100%|██████████| 10779/10779 [00:06<00:00, 1710.84it/s]

 Downloaded chef2


In [3]:
import os, shutil, yaml
from pathlib import Path

UNIFIED = {
    'glove': 0,   'gloves': 0,
    'no_glove': 1, 'no_gloves': 1,
    'hairnet': 2,  'hat': 2,
    'no_hairnet': 3, 'no-hairnet': 3,
    'mask': 4,    'maskon': 4,   'correct mask': 4,
    'no_mask': 5, 'maskoff': 5,
}
SKIP = {'clothes', 'human'}

OUT = Path("C:/Users/ziada/PycharmProjects/CVproject/merged5sets")
for split in ["train", "valid", "test"]:
    (OUT / split / "images").mkdir(parents=True, exist_ok=True)
    (OUT / split / "labels").mkdir(parents=True, exist_ok=True)

total = {s: 0 for s in ["train", "valid", "test"]}

for ds_path in download_paths:
    ds_path = Path(ds_path)
    with open(ds_path / "data.yaml") as f:
        meta = yaml.safe_load(f)
    local_classes = meta.get("names", [])

    id_map = {}
    for local_id, name in enumerate(local_classes):
        normalized = name.lower().strip()
        if normalized in SKIP:
            continue
        unified_id = UNIFIED.get(normalized)
        if unified_id is not None:
            id_map[local_id] = unified_id

    prefix = ds_path.name[:10]

    for split in ["train", "valid", "test"]:
        img_src = ds_path / split / "images"
        lbl_src = ds_path / split / "labels"
        if not img_src.exists():
            continue

        for img_file in img_src.iterdir():
            if img_file.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
                continue
            lbl_file = lbl_src / (img_file.stem + ".txt")
            if not lbl_file.exists():
                continue

            new_lines = []
            with open(lbl_file) as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    parts = line.split()
                    local_id = int(parts[0])
                    if local_id not in id_map:
                        continue
                    parts[0] = str(id_map[local_id])
                    new_lines.append(" ".join(parts))

            if not new_lines:
                continue

            new_stem = f"{prefix}_{img_file.stem}"
            shutil.copy(img_file, OUT / split / "images" / (new_stem + img_file.suffix))
            with open(OUT / split / "labels" / (new_stem + ".txt"), "w") as f:
                f.write("\n".join(new_lines))
            total[split] += 1

print("Merge complete")
for split, count in total.items():
    print(f"  {split}: {count} images")

Merge complete
  train: 10544 images
  valid: 2901 images
  test: 1480 images


In [5]:
data_yaml = {
    "path": "C:/Users/ziada/PycharmProjects/CVproject/merged5sets",
    "train": "train/images",
    "val":   "valid/images",
    "test":  "test/images",
    "nc": 6,
    "names": ["glove", "no_glove", "hairnet", "no_hairnet", "mask", "no_mask"]
}

with open(OUT / "data.yaml", "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print("data.yaml written")

data.yaml written


In [6]:
from ultralytics import YOLO

model = YOLO("yolo11m.pt")
model.train(
    data="C:/Users/ziada/PycharmProjects/CVproject/merged5sets/data.yaml",
    epochs=100,
    imgsz=640,
    batch=4,
    workers=2,
    name="merged_v5",
    project="C:/Users/ziada/PycharmProjects/CVproject/runs"
)

New https://pypi.org/project/ultralytics/8.4.77 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.33  Python-3.14.3 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:/Users/ziada/PycharmProjects/CVproject/merged5sets/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000019C1A608A60>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
 